## Tracing in OpeanAI Agents SDK

### What is Tracing

---
### 🔍 Tracing

- Provides complete **execution history** of an Agentic Workflow  
- Tracing is **enabled by default**  
- Tracing captures all events such as:
  - Agent run
  - LLM call
  - Tool call
  - Handoff
  - Guardrail, etc.
- Can be disabled using:
  - `RunConfig`
  - Environment variable:
    ```bash
    OPENAI_AGENTS_DISABLE_TRACING=1
    ```

---
### ⏱️ Span

- A **Span** represents a single **timed operation** inside a trace  
- Each span captures:
  - **Start time**
  - **End time**
- Spans can be **nested** (parent → child)
- Common span types include:
  - Agent Span
  - Function Span
  - Handoff Span
  - Guardrail Span
  - etc.
---

### RunConfig
Controls the runtime behavior of the agent.

### OpenAI Trace Example

In [9]:
from agents import Agent, Runner
from dotenv import load_dotenv
from agents.run import RunConfig

load_dotenv(override=True)

run_config = RunConfig(
    workflow_name="Net Tech Expert5",
    trace_id="trace_NetTechTrace"
    )

net_expert_instruction = """
You are a senior network engineer.
Explain the given Networking concept in single sentence
"""

network_expert_agent = Agent(
    name="Network Expert Agent",
    instructions=net_expert_instruction,
    model="gpt-5-nano"
    )

result = await Runner.run(network_expert_agent, 
                input="What is IP Address?", 
                run_config=run_config
                )
print(result.final_output)


An IP address is a unique numerical label assigned to a device’s network interface that identifies it on an IP network and enables routing of packets to and from that device.


In [1]:
from agents import Agent, Runner
from dotenv import load_dotenv
from agents.run import RunConfig

load_dotenv(override=True)

# run_config = RunConfig(
#     workflow_name="Net Expert",
#     # trace_id="trace_NetAgent"
#     )

net_expert_instruction = """
You are a senior network engineer.

Your domain of expertise is LIMITED to:
- Computer networks (LAN, WAN, routing, switching, firewalls, load balancers)
- Network protocols (TCP, UDP, IP, BGP, OSPF, HTTP, HTTPS, DNS, etc.)
- Network design, troubleshooting, and performance

HARD RULE:
- If the user asks ANYTHING that is NOT clearly about computer networking or network protocols,
  you MUST respond ONLY with exactly this sentence:
  "I don't know. I can only answer questions about computer networking."

- Do NOT try to be helpful outside your domain.
- Do NOT guess.
- Do NOT explain topics from physics, math, biology, history, programming (unless directly about networking), or any other area.
"""

network_expert_agent = Agent(
    name="Network Expert Agent",
    instructions=net_expert_instruction,
    model="gpt-5-nano"
    )

result = await Runner.run(network_expert_agent, 
                input="What is IP Address?", 
                # run_config=run_config
                )
print(result.final_output)


An IP address is a unique numerical identifier assigned to every device that participates in an IP network. It serves two main purposes: identifying the device (the host) and providing a location that helps route packets to that device.

Key points:
- Versions: IPv4 (32-bit, e.g., 192.168.1.10) and IPv6 (128-bit, e.g., 2001:0db8:85a3:0000:0000:8a2e:0370:7334).
- IPv4 structure: common to use a subnet mask or CIDR notation (e.g., 192.168.1.0/24) to separate the network part from the host part.
- How addresses are assigned: static (manual) or dynamic via DHCP (IPv4) or SLAAC/DHCPv6 (IPv6).
- Private vs public: private IPv4 ranges (e.g., 10.0.0.0/8, 172.16.0.0/12, 192.168.0.0/16) are not routable on the Internet; public addresses are assigned by an ISP. NAT is often used to share a single public address.
- How routing works: packets carry source and destination IPs; routers use the destination IP to forward the packet toward its destination.
- Local address resolution: IPv4 uses ARP to ma

## Simple Multi-Agent App

```mermaid
flowchart TD
    A["User Input<br/>Networking Concept<br/>(e.g., VLAN)"]
    
    subgraph T ["Tracing Enabled<br/>Net Topic Explainer"]
        direction TB
        B["Agent 1<br/>Prompt Generator Agent"]
        D["Agent 2<br/>Network Expert Agent"]
        E["Final Output<br/>Markdown Explanation + Configs"]
    end
    A --> B
    B --> D
    D --> E
```

In [10]:
from agents import Agent, Runner, trace
from dotenv import load_dotenv

load_dotenv(override=True)

prompt_generator_instruction = """
You are a Prompt Generator for network education.

Your task:
- The user gives a NETWORKING CONCEPT (example: VLAN)
- You must generate a CLEAR, STRUCTURED instruction
  that another agent can use to generate content.
- Instruct to make short and concise markdown responses.
- make sure you are replacing <concept> with the NETWORKING CONCEPT

STRICT RULES:
- Do NOT generate the actual explanation or configs
- Only generate instructions / prompt text
- Stay strictly within computer networking topics


The output MUST include the following sections:

1. What is <concept>
2. Advantages of <concept>
3. Configuration Example 1 (Cisco CLI)
4. Configuration Example 2 (Cisco CLI)
5. Verification / Show Commands

Write the output as a FINAL PROMPT that can be directly
passed to another agent.
"""

prompt_generator_agent = Agent(
    name="Prompt Generator Agent",
    instructions=prompt_generator_instruction,
    model="gpt-5-nano"
)

### Agent 2
net_expert_instruction = """
You are a senior network engineer.

You will be given a **networking topic name**.

Your expertise is STRICTLY LIMITED to:
- Computer networking concepts
- Switching and routing technologies
- VLANs and Layer 2 fundamentals
- Cisco IOS configuration and verification commands

Response Requirements:
- Generate the response ONLY in **Markdown format**
- Keep explanations technically accurate and concise

HARD RULE:
- If the given topic is NOT related to networking,
  respond ONLY with:
  "I don't know. I can only answer questions about computer networking."
"""
network_expert_agent = Agent(
    name="Network Expert Agent",
    instructions=net_expert_instruction,
    model="gpt-5-nano",
)

with trace(workflow_name="Net Topic Explainer2"):
    prompt_generator_output = await Runner.run(
        prompt_generator_agent, 
        input="VLAN")
    print("=== Agent-1 Output (Prompt Blueprint) ===")
    print(prompt_generator_output.final_output)

    net_expert_output = await Runner.run(
        network_expert_agent,
        input=prompt_generator_output.final_output,
    )

    print("=== Agent-2 Final Output (Generated Content) ===")
    print(net_expert_output.final_output)

=== Agent-1 Output (Prompt Blueprint) ===
FINAL PROMPT

You are an educational content generator for networking. Your task is to produce a CLEAR, STRUCTURED, and concise article about VLAN. Output must be in short, concise Markdown. Do not include content outside computer networking topics. Replace <concept> with VLAN wherever it appears in the prompt.

Sections (exact headings to include the content under):
1. What is VLAN
2. Advantages of VLAN
3. Configuration Example 1 (Cisco CLI)
4. Configuration Example 2 (Cisco CLI)
5. Verification / Show Commands

Guidelines for the content:
- Section 1: Provide a brief, high-level definition of VLAN, focusing on Layer 2 logical segmentation, broadcast domains, and isolation concepts. Keep to 2–4 bullet points or a short paragraph.
- Section 2: List the key advantages of using VLANs in a straightforward bullet format (e.g., reduced broadcasts, improved security, easier management, scalability, traffic engineering).
- Section 3: Configuration Exa